In [1]:
import newkernelo as lib
import numpy as np
import time
import os.path
import json
import logging
logging.getLogger().setLevel(logging.INFO)

In [2]:
# model =lib.FunctionalModel()
# help(lib.FunctionalModel)
# a = model.getDimensionY()
# print(model)

In [3]:
model = lib.TestModel()
print(model.__doc__)
print(lib.ShkuratovModel.__doc__)
# print(lib.FunctionalModel.__doc__)
# print(lib.ExternalPythonModel.__doc__)
# print(lib.ExternalPythonModel.__init__.__doc__)
help(lib.TestModel)




            The class :class:`TestModel` is low-dimensional functional model implemented in order to test the GLLiM method with simple but not trivial example.
            The functional F is designed so as to exhibit 2 solutions with D=9 and L=4. :math:`F = A ◦ G ◦ H`, where
             - :math:`A` is a (DxL) injective matrix,
            
                .. math::
                    A = \frac{1}{2} \begin{pmatrix}
                        1 & 2 & 2 & 1 \\
                        0 & 0.5 & 0 & 0 \\
                        0 & 0 & 1 & 0 \\
                        0 & 0 & 0 & 3 \\
                        0.2 & 0 & 0 & 0 \\
                        0 & -0.5 & 0 & 0 \\
                        -0.2 & 0 & -1 & 0 \\
                        -1 & 0 & 2 & 0 \\
                        0 & 0 & 0 & -0.7
                    \end{pmatrix}

             - :math:`G(x)` = :math:`(exp(x_1), exp(x_2), exp(x_3), exp(x_4))` and 
             - :math:`H(x)` = :math:`(x_1, x_2, 4(x_3-0.5)^2, x_4)`.
       

## Test Model

In [4]:
model = lib.TestModel()
e = np.exp(1)
x_test = np.zeros(4)
y_test = model.F(x_test)
true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
print("y_test")
print(y_test)
print("true_result")
print(true_result)

y_test
[ 4.71828183  0.25        1.35914091  1.5         0.1        -0.25
 -1.45914091  2.21828183 -0.35      ]
true_result
[ 4.71828183  0.25        1.35914091  1.5         0.1        -0.25
 -1.45914091  2.21828183 -0.35      ]


In [5]:
e = np.exp(1)
x = np.ones(4)

# y = np.zeros(9)
# t0 = time.time()
# for i in range(10000000): #10000000
#     model.F(x,y)
# tf = time.time() - t0
# print("Time F by reference: {}".format(tf))

# t0 = time.time()
# for i in range(10000000):
#     z = model.F(x)
# tf = time.time() - t0
# print("Time F by value: {}".format(tf))

# true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
# print("true_result")
# print(true_result)

z = model.F(x)
print(z)
print(x)
print(x.shape)
model.toPhysic(x) # does nothing
print(x)
print(x.shape)
w = model.toPhysic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
model.fromPhysic(x)
print(x)
y = model.fromPhysic(x)
print(x)
print(y)
z = model.fromPhysic(y)
print(x)
print(y)
print(z)

[ 8.15484549  0.67957046  1.35914091  4.07742274  0.27182818 -0.67957046
 -1.6309691   1.35914091 -0.95139864]
[1. 1. 1. 1.]
(4,)
[1. 1. 1. 1.]
(4,)
[1. 1. 1. 1.]
[2. 2. 2. 2.]
(4,)
=========== From physic ==========
[1. 1. 1. 1.]
[1. 1. 1. 1.]
[1. 1. 1. 1.]
[0.5 0.5 0.5 0.5]
[1. 1. 1. 1.]
[0.5 0.5 0.5 0.5]
[0.25 0.25 0.25 0.25]


## Shkuratov

In [6]:
print(os.getcwd())
with open('../../pytest/new_test_shkuratov.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]


# # Create JSON file with geometries
geometries = np.empty((row_size,col_size))
var_geom = ["inc", "eme", "phi"]
for j in range(3):
    i=0
    for v in data[var_geom[j]]:
        geometries[i,j] = v # geometries.shape = (D,3)
        i+=1
# print(geometries)
print(geometries.shape)

# geom = {'geometries': [[geometries[j,:].tolist()] for j in range(3)]
# }
# with open('geometries_shkuratov.json', 'w') as fp:
#     json.dump(geom, fp)

## INTEGRATION au code C++
variant = "5p"
physicalModel = lib.ShkuratovModel(geometries, variant, scalingCoeffs, offset)


### TEST
N = 100
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])



# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = physicalModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])


/home/luc/Documents/dev/kernelo-gllim-is/new_kernelo/python
(50, 3)
(50,)
[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
False
True
[0.01652884 0.00787757 0.023348   0.01033272 0.00816459 0.02172611
 0.0126815  0.04004887 0.01367903 0.02221388]
[0.01652884 0.00787757 0.023348   0.01033272 0.00816459 0.02172611
 0.0126815  0.04004887 0.01367903 0.02221388]


## Hapke

In [7]:
with open('../../data_ref_hapke6p_geom70.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "six", 30.0,1,0)
print(hapkeModel.getDimensionY())
print(hapkeModel.getDimensionX())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n])
print(result)

(70, 3)
(20000, 6)
(20000, 70)
70
6
(70,)
[True, True, True, True, True, True, True, True, True, True]
False
True
[0.661713   0.71376229 0.75547619 0.79594483 0.84898809 0.94010314
 1.09388016 1.09388016 0.94010314 0.84898809 0.79594483 0.75547619
 0.71376229 0.661713   0.67201986 0.71167323 0.74084312 0.76380446
 0.78506504 0.81104192 0.85408973 0.94010314 1.09745333 1.10620349
 0.93548503 0.82486549 0.74998315 0.68140027 0.79251966 0.94548664
 1.15975826 1.12410881 0.93548503 0.83902679 0.79594483 0.77532515
 0.76380446 0.75567048 0.74812848 0.73961698 0.72954369 0.71933715
 0.89568301 0.79645035 0.75188174 0.72954369 0.71763    0.71167323
 0.71025644 0.71376229 0.72469036 0.74998315 0.80807864 0.94548664
 1.23267475 1.39492325 0.67631355 0.73944407 0.8019977  0.88319617
 0.99919454 1.055518   0.94010314 0.85957158 0.81506496 0.78747365
 0.7650571  0.74117638 0.71104034 0.67001222]
[0.661713   0.71376229 0.75547619 0.79594483 0.84898809 0.94010314
 1.09388016 1.09388016 0.94010314 0.

In [8]:
with open('../../data_ref_hapke4p_geom70.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "four", 30.0,1,0)
print(hapkeModel.getDimensionY())
print(hapkeModel.getDimensionX())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n,:10])
print(result[:10])

### NOTE : le expected_results[n][2] et result[2] sont différents.... mais les autres sont égaux. à voir...

(70, 3)
(20000, 4)
(20000, 70)
70
4
(70,)
[True, True, True, True, True, True, True, True, True, True]
False
True
[0.65998213 0.71192246 0.75322617 0.79272546 0.84355547 0.92952621
 1.07279062 1.07279062 0.92952621 0.84355547]
[0.65998213 0.71192246 0.75322617 0.79272546 0.84355547 0.92952621
 1.07279062 1.07279062 0.92952621 0.84355547]


In [9]:
print(hapkeModel.getDimensionY())
print(hapkeModel.getDimensionX())
x = np.ones(hapkeModel.getDimensionX()) / 10

print(x)
print(x.shape)
hapkeModel.toPhysic(x) # does nothing
print(x)
print(x.shape)
w = hapkeModel.toPhysic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
hapkeModel.fromPhysic(x)
print(x)
y = hapkeModel.fromPhysic(x)
print(x)
print(y)
z = hapkeModel.fromPhysic(y)
print(x)
print(y)
print(z)

70
4
[0.1 0.1 0.1 0.1]
(4,)
[0.1 0.1 0.1 0.1]
(4,)
[0.1 0.1 0.1 0.1]
[0.19 3.   0.1  0.1 ]
(4,)
=========== From physic ==========
[0.1 0.1 0.1 0.1]
[0.1 0.1 0.1 0.1]
[0.1 0.1 0.1 0.1]
[0.0513167  0.00333333 0.1        0.1       ]
[0.1 0.1 0.1 0.1]
[0.0513167  0.00333333 0.1        0.1       ]
[0.02599625 0.00011111 0.1        0.1       ]


## External model

In [10]:
externalPythonModel = lib.ExternalPythonModel("ShkuratovModel5p", "ShkuratovModel5pPython", "../cpptest/functionalModel/dataRef/")

print(externalPythonModel.getDimensionY())
print(externalPythonModel.getDimensionX())

50
5


In [11]:
x = np.ones(externalPythonModel.getDimensionX())

print(x)
print(x.shape)
print("=========== To physic ==========")
externalPythonModel.toPhysic(x) # does nothing
print(x)
print(x.shape)
w = externalPythonModel.toPhysic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
externalPythonModel.fromPhysic(x)
print(x)
y = externalPythonModel.fromPhysic(x)
print(x)
print(y)
z = externalPythonModel.fromPhysic(y)
print(x)
print(y)
print(z)

[1. 1. 1. 1. 1.]
(5,)
=========== To physic ==========
[1. 1. 1. 1. 1.]
(5,)
[1. 1. 1. 1. 1.]
[1.  1.5 1.7 1.5 1.5]
(5,)
=========== From physic ==========
[1. 1. 1. 1. 1.]
[1. 1. 1. 1. 1.]
[1.         0.66666667 0.53333333 0.66666667 0.66666667]
[1. 1. 1. 1. 1.]
[1.         0.66666667 0.53333333 0.66666667 0.66666667]
[1.         0.44444444 0.22222222 0.44444444 0.44444444]


In [12]:
with open('../../pytest/new_test_shkuratov.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]



### TEST
N = 3
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])


# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = externalPythonModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])

(50,)
[True, True, True]
False
True
[0.16476508 0.0988973  0.21693669 0.1171771  0.1004486  0.19843122
 0.13557557 0.30234728 0.14297123 0.20419017]
[0.16476508 0.0988973  0.21693669 0.1171771  0.1004486  0.19843122
 0.13557557 0.30234728 0.14297123 0.20419017]


## dataGeneration

In [13]:
model = lib.TestModel()

# covariances = np.random.uniform(0, 0.000001, model.getDimensionY())
# stat_model = lib.GaussianStatModel("sobol", model, covariances, 12345)

r = 100 # SNR => sigma = Signal/r
stat_model = lib.DependentGaussianStatModel("sobol", model, r, 12345)

In [14]:
x_gen, y_gen = stat_model.gen_data(10)
print(x_gen)
print(y_gen)

[[0.5    0.5    0.5    0.5   ]
 [0.75   0.25   0.25   0.25  ]
 [0.25   0.75   0.75   0.75  ]
 [0.375  0.375  0.625  0.875 ]
 [0.875  0.875  0.125  0.375 ]
 [0.625  0.125  0.875  0.625 ]
 [0.125  0.625  0.375  0.125 ]
 [0.1875 0.3125 0.9375 0.4375]
 [0.6875 0.8125 0.4375 0.9375]
 [0.9375 0.0625 0.6875 0.1875]]
[[ 4.24680063  0.40523434  0.50289063  2.5184103   0.16260053 -0.41362676
  -0.65524854  0.17537323 -0.58777066]
 [ 4.21426282  0.32035536  0.64776598  1.92065404  0.20995966 -0.32268618
  -0.85575377  0.2284948  -0.44784703]
 [ 5.15435173  0.52908254  0.63534359  3.16940687  0.12884647 -0.5317442
  -0.77798519  0.65076844 -0.73360808]
 [ 4.43731737  0.36305403  0.52707526  3.64514124  0.14361027 -0.35579802
  -0.68097467  0.3297535  -0.8390907 ]
 [ 6.19679477  0.59812534  0.87614486  2.16666344  0.24127145 -0.59324592
  -1.11500258  0.55709649 -0.50902197]
 [ 4.68854822  0.28291444  0.88322964  2.78831128  0.18768088 -0.28531561
  -1.06376379  0.82271813 -0.65359927]
 [ 4.0583422

In [15]:
print(x_gen.shape)
print(y_gen.shape)

(10, 4)
(10, 9)


In [16]:
for n in range(x_gen.shape[0]):
    result = model.F(x_gen[n,:])
    print(np.allclose(y_gen[n,:], result, rtol=1e-3))

False
False
False
False
False
False
False
False
False
False


### New FunctionalModel.dataGen()

In [24]:
x_gen, y_gen = model.genData(10, "sobol", np.ones(model.getDimensionY()), 1234)

In [25]:
print(x_gen.shape)
print(y_gen.shape)

[[0.5    0.5    0.5    0.5   ]
 [0.75   0.25   0.25   0.25  ]
 [0.25   0.75   0.75   0.75  ]
 [0.375  0.375  0.625  0.875 ]
 [0.875  0.875  0.125  0.375 ]
 [0.625  0.125  0.875  0.625 ]
 [0.125  0.625  0.375  0.125 ]
 [0.1875 0.3125 0.9375 0.4375]
 [0.6875 0.8125 0.4375 0.9375]
 [0.9375 0.0625 0.6875 0.1875]]
[[ 4.50259668e+00  7.59885854e-02  9.36289255e-01  2.45559387e+00
   5.02916301e-01 -8.89005783e-01 -4.12213248e-02 -9.42841950e-01
   3.07709119e-02]
 [ 3.96009912e+00  5.56836879e-01  2.55429206e-02  1.78375570e+00
   2.49721363e-01  2.57987075e-01 -1.40170966e+00  2.08570681e+00
   4.46837109e-01]
 [ 2.75731948e+00  1.36531311e+00  1.96436888e+00  2.36350082e+00
   1.33221853e+00 -2.40468294e-01 -4.11605915e-02 -1.29801865e-01
  -1.90138121e+00]
 [ 3.10315611e+00 -4.47449691e-03  3.88091688e-01  4.24317603e+00
   4.80964499e-01 -1.08083380e-01 -1.53424561e-01 -1.42963708e-01
  -1.48110274e+00]
 [ 5.44224551e+00  2.87187292e-01  1.33738455e+00  1.79772647e+00
   1.88503308e+00  

In [26]:
x_gen, y_gen = model.genData(10, "sobol", 0.1, 1234)

genData with double


In [27]:
print(x_gen.shape)
print(y_gen.shape)

[[0.5    0.5    0.5    0.5   ]
 [0.75   0.25   0.25   0.25  ]
 [0.25   0.75   0.75   0.75  ]
 [0.375  0.375  0.625  0.875 ]
 [0.875  0.875  0.125  0.375 ]
 [0.625  0.125  0.875  0.625 ]
 [0.125  0.625  0.375  0.125 ]
 [0.1875 0.3125 0.9375 0.4375]
 [0.6875 0.8125 0.4375 0.9375]
 [0.9375 0.0625 0.6875 0.1875]]
[[  13.11382381   -0.97353583    2.68144628    2.04058857    0.72221275
     1.5532004    -4.81135248   -1.78885411   -4.08451198]
 [  -8.89843687    1.07803732   -3.31580167   -0.81437556    0.29219122
    -2.17961205    3.82460694    4.42070711   -4.47721817]
 [-114.48967147    4.95411401    9.13170739  -22.60953493    1.67413287
    -2.05762721   -6.38870435   -4.31313494    7.85726505]
 [ -55.28069045   -0.97565304   -0.23501664   26.80250509    0.63359836
    -1.29372189   -4.23131841   -1.28046875    4.54643803]
 [ -32.75261191   -1.2745916     4.91290008   -6.21486459    4.18638654
   -10.91575156    9.60190502   10.8189392     1.56694434]
 [  14.02646093    0.20814998    4